# PTCG Imitation-Learning Submission

Attach one or more trained checkpoints from `training/train.py` and a Dataset containing `cg`. Edit only `ENSEMBLE_ENABLED`, `DAMAGE_COUNTER_KO_MASK`, `MODEL_PATHS`, `CG_PATH`, and `DECK` in the parameter cell. Every model architecture is restored automatically from its checkpoint. Running all cells creates `/kaggle/working/submission.tar.gz`.

In [ ]:
import json
from pathlib import Path
import torch

# Edit these paths after attaching your Kaggle Datasets.
ENSEMBLE_ENABLED = False
DAMAGE_COUNTER_KO_MASK = True
MODEL_PATHS = [
    Path('/kaggle/input/your-model-dataset/epoch-003.pt'),
]
CG_PATH = Path('/kaggle/input/your-cg-dataset/cg')

# This is the deck used by the submitted agent, not a model-architecture setting.
DECK = [7, 7, 7, 7, 7, 7, 7, 7, 7, 7,
       104, 104, 112, 112, 112, 112,
       646, 646, 646, 646, 647, 647, 647, 648, 648, 648,
       860, 860, 1079, 1079, 1079, 1080,
       1086, 1086, 1086, 1086, 1097, 1097, 1097, 1122, 1137,
       1152, 1152, 1152, 1152, 1182, 1182,
       1219, 1219, 1219, 1219, 1227, 1227, 1227, 1227,
       1231, 1259, 1259, 1259, 1259]
assert len(DECK) == 60
Path('deck.csv').write_text('\n'.join(map(str, DECK)) + '\n')

assert type(ENSEMBLE_ENABLED) is bool, 'ENSEMBLE_ENABLED must be boolean'
assert type(DAMAGE_COUNTER_KO_MASK) is bool, 'DAMAGE_COUNTER_KO_MASK must be boolean'
assert all(isinstance(path, Path) for path in MODEL_PATHS), 'MODEL_PATHS must contain Path values'
assert len(set(MODEL_PATHS)) == len(MODEL_PATHS), 'MODEL_PATHS must be distinct'
if ENSEMBLE_ENABLED:
    assert len(MODEL_PATHS) >= 2, 'Enabled ensemble requires at least two checkpoints'
else:
    assert len(MODEL_PATHS) == 1, 'Disabled ensemble requires exactly one checkpoint'
for model_path in MODEL_PATHS:
    assert model_path.is_file(), f'Checkpoint not found: {model_path}'
assert CG_PATH.is_dir(), f'cg directory not found: {CG_PATH}'
assert (CG_PATH / '__init__.py').is_file(), f'Not a cg package: {CG_PATH}'

def load_checkpoint_preview(path):
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location='cpu')
    assert isinstance(checkpoint, dict), 'Checkpoint must be a mapping'
    assert 'model' in checkpoint and 'config' in checkpoint, (
        'Checkpoint must contain model and config keys'
    )
    return checkpoint

architectures = []
for model_index, model_path in enumerate(MODEL_PATHS, start=1):
    checkpoint_preview = load_checkpoint_preview(model_path)
    architecture = checkpoint_preview['config']
    architectures.append(architecture)
    print(
        f'Model {model_index}: {model_path} | d_model={architecture["d_model"]} ' 
        f'heads={architecture["num_heads"]} encoder={architecture["encoder_layers"]} ' 
        f'decoder={architecture["decoder_layers"]} norm={architecture["norm_mode"]}'
    )
    del checkpoint_preview

compatibility_keys = ('card_count', 'attack_count', 'encoder_size')
reference = {key: architectures[0][key] for key in compatibility_keys}
for model_index, architecture in enumerate(architectures[1:], start=2):
    differences = [key for key in compatibility_keys if architecture[key] != reference[key]]
    assert not differences, f'Model {model_index} is incompatible: {differences}'

MODEL_ARCHIVE_NAMES = [
    f'model-{index:03d}.pt' for index in range(1, len(MODEL_PATHS) + 1)
]
MODEL_MANIFEST = {
    'ensemble_enabled': ENSEMBLE_ENABLED,
    'damage_counter_ko_mask': DAMAGE_COUNTER_KO_MASK,
    'model_files': MODEL_ARCHIVE_NAMES,
}
Path('model_manifest.json').write_text(
    json.dumps(MODEL_MANIFEST, indent=2), encoding='utf-8'
)
print('Ensemble:', ENSEMBLE_ENABLED, 'models:', len(MODEL_PATHS))
print('Damage-counter KO mask:', DAMAGE_COUNTER_KO_MASK)
print('cg:', CG_PATH)
print('Deck cards:', len(DECK))

In [ ]:
%%writefile main.py
from __future__ import annotations

import json
import os
from collections import Counter, deque
from dataclasses import dataclass, field
from itertools import combinations
from typing import Any

import torch
from cg.api import AreaType, OptionType, all_attack, all_card_data, to_observation_class

torch.set_num_threads(1)
MAX_ACTIONS = 64
ENCODER_TOKENS = 28
POKEMON_ENCODER_TOKENS = 18
BENCH_SLOTS = 8
PLAYER_BENCH_COUNT_INDEX = 10
OWN_SUMMARY_DIM = 69
OPPONENT_SUMMARY_DIM = 71
GLOBAL_SUMMARY_DIM = 73
SELECT_TYPE_DIM = 11
SELECT_CONTEXT_DIM = 49
CARD_FEATURE_DIM = 54
CARD_TYPE_DIM = 7
ENERGY_TYPE_DIM = 12
CARD_ENERGY_TYPE_OFFSET = 7
CARD_HP_INDEX = 19
CARD_RETREAT_INDEX = 20
CARD_WEAKNESS_OFFSET = 21
CARD_RESISTANCE_OFFSET = 34
CARD_STAGE_OFFSET = 47
CARD_SPECIAL_OFFSET = 50
CARD_WEAKNESS_DIM = 13
CARD_RESISTANCE_DIM = 13
OPTION_TYPE_COUNT = 17
OPTION_CONTEXT_COUNT = 49
OPTION_VALUE_COUNT = 63
OPTION_PLAYER_RELATION_COUNT = 3
OPTION_AREA_COUNT = 13
OPTION_SPECIAL_CONDITION_COUNT = 6
POKEMON_DYNAMIC_WORD_DIM = 23
POKEMON_DYNAMIC_DIM = 46
ATTACK_DYNAMIC_DIM = 6
OPTION_NUMERIC_DIM = 5
ATTACK_FEATURE_DIM = 14
HISTORY_STEPS = 3
HISTORY_STRUCTURAL_DIM = 8
HISTORY_OPTION_TYPE_INDEX = 0
HISTORY_SOURCE_AREA_INDEX = 1
HISTORY_TARGET_AREA_INDEX = 2
HISTORY_SOURCE_RELATION_INDEX = 3
HISTORY_TARGET_RELATION_INDEX = 4
HISTORY_NUMBER_INDEX = 5
HISTORY_COUNT_INDEX = 6
HISTORY_SPECIAL_CONDITION_INDEX = 7
OPTION_PLAYER_RELATION_INDEX = 7
OPTION_AREA_INDEX = 8
OPTION_IN_PLAY_AREA_INDEX = 9
OPTION_TYPE_PLAY = 7
OPTION_TYPE_ATTACH = 8
OPTION_TYPE_EVOLVE = 9
OPTION_TYPE_RETREAT = 12
CARD_REGION_NAMES = (
    'own_bench', 'opponent_bench', 'own_active', 'opponent_active',
    'own_discard', 'opponent_discard', 'own_hand', 'opponent_hand',
    'own_deck', 'opponent_deck', 'own_prize', 'opponent_prize',
    'stadium', 'looking', 'unknown',
)
CARD_REGION_INDEX = {name: index for index, name in enumerate(CARD_REGION_NAMES)}
OWN_AREA_REGIONS = (
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['own_deck'],
    CARD_REGION_INDEX['own_hand'], CARD_REGION_INDEX['own_discard'],
    CARD_REGION_INDEX['own_active'], CARD_REGION_INDEX['own_bench'],
    CARD_REGION_INDEX['own_prize'], CARD_REGION_INDEX['stadium'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['looking'],
)
OPPONENT_AREA_REGIONS = (
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['opponent_deck'],
    CARD_REGION_INDEX['opponent_hand'], CARD_REGION_INDEX['opponent_discard'],
    CARD_REGION_INDEX['opponent_active'], CARD_REGION_INDEX['opponent_bench'],
    CARD_REGION_INDEX['opponent_prize'], CARD_REGION_INDEX['stadium'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['unknown'], CARD_REGION_INDEX['unknown'],
    CARD_REGION_INDEX['looking'],
)


def optional_log_int(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


class RevealedHandTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.cards = ({}, {})

    def update(self, logs):
        for log in logs or ():
            log_type = optional_log_int(getattr(log, 'type', None))
            player = optional_log_int(getattr(log, 'playerIndex', None))
            if player not in (0, 1) or log_type is None:
                continue
            known = self.cards[player]
            serial = optional_log_int(getattr(log, 'serial', None))
            if log_type == 6:
                source = optional_log_int(getattr(log, 'fromArea', None))
                target = optional_log_int(getattr(log, 'toArea', None))
                if source == 2:
                    if serial is None:
                        known.clear()
                    else:
                        known.pop(serial, None)
                if target == 2:
                    card_id = optional_log_int(getattr(log, 'cardId', None))
                    if serial is not None and card_id is not None and card_id >= 0:
                        known[serial] = card_id
            elif log_type == 7:
                if optional_log_int(getattr(log, 'fromArea', None)) == 2:
                    known.clear()
            elif log_type in (10, 11, 12):
                if serial is None:
                    known.clear()
                else:
                    known.pop(serial, None)

    def relative_cards(self, your_index):
        yours = int(your_index)
        if yours not in (0, 1):
            raise ValueError('your_index must be 0 or 1')
        def ordered(player):
            return [card_id for _, card_id in sorted(self.cards[player].items())]
        return ordered(yours), ordered(1 - yours)


def asset_path(name: str) -> str:
    # Kaggle executes main.py with exec(), so __file__ is not guaranteed.
    candidates = [name, os.path.join('/kaggle_simulations/agent', name)]
    module_file = globals().get('__file__')
    if module_file:
        candidates.append(os.path.join(os.path.dirname(module_file), name))
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(name)


def read_deck() -> list[int]:
    with open(asset_path('deck.csv'), encoding='utf-8') as handle:
        deck = [int(line.strip()) for line in handle if line.strip()]
    if len(deck) != 60:
        raise ValueError(f'Expected 60 cards, found {len(deck)}')
    return deck


MY_DECK = read_deck()


def build_card_feature_table(cards, card_count):
    features = torch.zeros((card_count, CARD_FEATURE_DIM), dtype=torch.float32)
    for card in cards:
        card_id = int(card.cardId)
        if not 0 <= card_id < card_count:
            continue
        card_type = int(card.cardType)
        if 0 <= card_type < CARD_TYPE_DIM:
            features[card_id, card_type] = 1
        energy_type = int(card.energyType)
        if 0 <= energy_type < ENERGY_TYPE_DIM:
            features[card_id, CARD_ENERGY_TYPE_OFFSET + energy_type] = 1
        features[card_id, CARD_HP_INDEX] = float(card.hp) / 400
        features[card_id, CARD_RETREAT_INDEX] = float(card.retreatCost) / 5
        weakness = None if card.weakness is None else int(card.weakness)
        weakness = weakness if weakness is not None and 0 <= weakness < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_WEAKNESS_OFFSET + weakness] = 1
        resistance = None if card.resistance is None else int(card.resistance)
        resistance = resistance if resistance is not None and 0 <= resistance < ENERGY_TYPE_DIM else ENERGY_TYPE_DIM
        features[card_id, CARD_RESISTANCE_OFFSET + resistance] = 1
        features[card_id, CARD_STAGE_OFFSET:CARD_SPECIAL_OFFSET] = torch.tensor([
            float(card.basic), float(card.stage1), float(card.stage2),
        ])
        features[card_id, CARD_SPECIAL_OFFSET:CARD_FEATURE_DIM] = torch.tensor([
            float(card.ex), float(card.megaEx), float(card.tera), float(card.aceSpec),
        ])
    return features


def build_attack_feature_table(attacks, attack_count):
    features = torch.zeros((attack_count, ATTACK_FEATURE_DIM), dtype=torch.float32)
    for attack in attacks:
        attack_id = int(attack.attackId)
        if not 0 <= attack_id < attack_count:
            continue
        features[attack_id, 0] = float(attack.damage) / 300.0
        energies = list(attack.energies or [])
        for energy in energies:
            energy_type = int(energy)
            if 0 <= energy_type < ENERGY_TYPE_DIM:
                features[attack_id, 1 + energy_type] += 1
        features[attack_id, -1] = len(energies) / 5.0
    return features


@dataclass(frozen=True)
class ModelConfig:
    card_count: int
    attack_count: int
    encoder_size: int = 25_000
    d_model: int = 128
    num_heads: int = 2
    d_feedforward: int = 256
    encoder_layers: int = 1
    decoder_layers: int = 1
    norm_mode: str = 'postnorm'
    transformer_activation: str = 'relu'
    transformer_dropout: float = 0.0
    dropout_embedding: bool = False
    dropout_attention_probs: bool = False
    dropout_attention_output: bool = False
    dropout_ffn_output: bool = False
    summary_mlp_layers: int = 1
    card_mlp_layers: int = 1
    option_numeric_mlp_layers: int = 1
    option_token_mlp_layers: int = 0
    card_mlp_scope: str = 'shared'
    pokemon_appear_embedding: bool = False
    bench_token_mlp_layers: int = 0
    active_token_mlp_layers: int = 0
    discard_token_mlp_layers: int = 0
    hand_token_mlp_layers: int = 0
    deck_token_mlp_layers: int = 0
    revealed_hand_token_mlp_layers: int = 0
    learnable_cls_token: bool = False
    region_token_mlp_residual: bool = True
    history_encoding: str = 'off'
    history_action_mlp_layers: int = 1
    history_sequence_mlp_layers: int = 2

    def __post_init__(self):
        if self.transformer_activation not in {'relu', 'gelu', 'geglu'}:
            raise ValueError('invalid transformer_activation')
        if not 0.0 <= float(self.transformer_dropout) < 1.0:
            raise ValueError('transformer_dropout must be in [0, 1)')
        if self.card_mlp_scope not in {'shared', 'region'}:
            raise ValueError('card_mlp_scope must be shared or region')
        if self.option_numeric_mlp_layers < 1:
            raise ValueError('option_numeric_mlp_layers must be >= 1')
        if self.option_token_mlp_layers < 0:
            raise ValueError('option_token_mlp_layers must be >= 0')
        if self.history_encoding not in {'off', 'basic', 'structural', 'full'}:
            raise ValueError('invalid history_encoding')
        if self.history_action_mlp_layers < 0:
            raise ValueError('history_action_mlp_layers must be >= 0')
        if self.history_encoding != 'off' and self.history_sequence_mlp_layers < 1:
            raise ValueError('enabled history requires history_sequence_mlp_layers >= 1')
        for name in (
            'bench_token_mlp_layers', 'active_token_mlp_layers',
            'discard_token_mlp_layers', 'hand_token_mlp_layers',
            'deck_token_mlp_layers', 'revealed_hand_token_mlp_layers',
        ):
            if getattr(self, name) < 0:
                raise ValueError(f'{name} must be >= 0')
        if type(self.learnable_cls_token) is not bool:
            raise ValueError('learnable_cls_token must be a boolean')


def projection_mlp(input_dim, d_model, layers):
    if layers < 1:
        raise ValueError('projection MLP must contain at least one layer')
    modules = [torch.nn.Linear(input_dim, d_model)]
    for _ in range(layers - 1):
        modules.extend([torch.nn.ReLU(), torch.nn.Linear(d_model, d_model)])
    return modules[0] if len(modules) == 1 else torch.nn.Sequential(*modules)


def fill_card_range(card_mapping, region_mapping, start, card_count, region):
    end = start + card_count
    card_mapping[start:end] = torch.arange(card_count)
    region_mapping[start:end] = region
    return end


def encoder_card_mappings(config):
    card_mapping = torch.full((config.encoder_size,), config.card_count, dtype=torch.long)
    region_mapping = torch.full(
        (config.encoder_size,), CARD_REGION_INDEX['unknown'], dtype=torch.long
    )
    position = 0
    field_regions = (
        CARD_REGION_INDEX['own_bench'], CARD_REGION_INDEX['opponent_bench'],
        CARD_REGION_INDEX['own_active'], CARD_REGION_INDEX['opponent_active'],
    )
    for region in field_regions:
        position += 2
        for _ in range(3):
            position = fill_card_range(
                card_mapping, region_mapping, position, config.card_count, region
            )
    zone_regions = (
        CARD_REGION_INDEX['own_discard'], CARD_REGION_INDEX['opponent_discard'],
        CARD_REGION_INDEX['own_hand'], CARD_REGION_INDEX['own_deck'],
        CARD_REGION_INDEX['stadium'], CARD_REGION_INDEX['own_hand'],
        CARD_REGION_INDEX['opponent_hand'],
    )
    for region in zone_regions:
        position = fill_card_range(
            card_mapping, region_mapping, position, config.card_count, region
        )
    return card_mapping, region_mapping


class CardAwareEmbeddingBag(torch.nn.EmbeddingBag):
    def __init__(
        self, num_embeddings, embedding_dim, index_to_card_id, index_to_card_region
    ):
        super().__init__(num_embeddings, embedding_dim, mode='sum')
        self.register_buffer('index_to_card_id', index_to_card_id, persistent=False)
        self.register_buffer(
            'index_to_card_region', index_to_card_region, persistent=False
        )

    def forward(self, indices, offsets, weights, projected_card_features):
        learned = super().forward(indices, offsets, per_sample_weights=weights)
        card_ids = self.index_to_card_id[indices]
        if projected_card_features.ndim == 2:
            static_indices = card_ids
            static_table = projected_card_features
        else:
            region_ids = self.index_to_card_region[indices]
            card_vocabulary = projected_card_features.size(1)
            static_indices = region_ids * card_vocabulary + card_ids
            static_table = projected_card_features.flatten(0, 1)
        static_weights = None if weights is None else weights.to(projected_card_features.dtype)
        static = torch.nn.functional.embedding_bag(
            static_indices, static_table, offsets,
            mode='sum', per_sample_weights=static_weights,
        )
        return learned + static


class EncoderLayer(torch.nn.TransformerEncoderLayer):
    def __init__(self, d_model, num_heads, d_feedforward, norm_mode, activation='relu',
                 dropout=0.0, dropout_attention_probs=False,
                 dropout_attention_output=False, dropout_ffn_output=False):
        super().__init__(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_feedforward,
            dropout=0.0, activation='relu', norm_first=norm_mode == 'prenorm',
        )
        self.transformer_activation = activation
        if activation == 'geglu':
            self.linear1 = torch.nn.Linear(d_model, 2 * d_feedforward)
        self.activation = self._activate
        self.activation_relu_or_gelu = 1 if activation == 'relu' else 0
        probability = float(dropout)
        self.self_attn.dropout = probability if dropout_attention_probs else 0.0
        self.dropout.p = 0.0
        self.dropout1.p = probability if dropout_attention_output else 0.0
        self.dropout2.p = probability if dropout_ffn_output else 0.0

    def _activate(self, value):
        if self.transformer_activation == 'relu':
            return torch.nn.functional.relu(value)
        if self.transformer_activation == 'gelu':
            return torch.nn.functional.gelu(value, approximate='tanh')
        value, gate = value.chunk(2, dim=-1)
        return value * torch.nn.functional.gelu(gate, approximate='tanh')


class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_feedforward, norm_mode, activation='relu',
                 dropout=0.0, dropout_attention_probs=False,
                 dropout_attention_output=False, dropout_ffn_output=False):
        super().__init__()
        self.prenorm = norm_mode == 'prenorm'
        probability = float(dropout)
        self.transformer_activation = activation
        self.attention = torch.nn.MultiheadAttention(
            d_model, num_heads,
            dropout=probability if dropout_attention_probs else 0.0,
        )
        self.fc1 = torch.nn.Linear(
            d_model, d_feedforward * (2 if activation == 'geglu' else 1)
        )
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.attention_output_dropout = torch.nn.Dropout(
            probability if dropout_attention_output else 0.0
        )
        self.ffn_output_dropout = torch.nn.Dropout(
            probability if dropout_ffn_output else 0.0
        )

    def _activate(self, value):
        if self.transformer_activation == 'relu':
            return torch.nn.functional.relu(value)
        if self.transformer_activation == 'gelu':
            return torch.nn.functional.gelu(value, approximate='tanh')
        value, gate = value.chunk(2, dim=-1)
        return value * torch.nn.functional.gelu(gate, approximate='tanh')

    def _feed_forward(self, value):
        return self.ffn_output_dropout(self.fc2(self._activate(self.fc1(value))))

    def forward(self, x, encoder_out, encoder_padding_mask):
        if self.prenorm:
            query = self.norm1(x)
            attended, _ = self.attention(
                query, encoder_out, encoder_out,
                key_padding_mask=encoder_padding_mask, need_weights=False
            )
            x = x + self.attention_output_dropout(attended)
            return x + self._feed_forward(self.norm2(x))
        y, _ = self.attention(
            x, encoder_out, encoder_out,
            key_padding_mask=encoder_padding_mask, need_weights=False
        )
        residual = self.norm1(x + self.attention_output_dropout(y))
        return self.norm2(residual + self._feed_forward(residual))


class PTCGTransformer(torch.nn.Module):
    def __init__(self, config, card_feature_table, attack_feature_table):
        super().__init__()
        self.config = config
        self.encoder_token_count = (ENCODER_TOKENS + int(config.history_encoding != 'off')
                                    + int(config.learnable_cls_token))
        self.register_buffer('card_feature_table', card_feature_table)
        self.register_buffer('attack_feature_table', attack_feature_table)
        self.card_feature_projection = None
        self.card_feature_projections = None
        if config.card_mlp_layers > 0 and config.card_mlp_scope == 'shared':
            self.card_feature_projection = projection_mlp(
                CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers
            )
        elif config.card_mlp_layers > 0:
            self.card_feature_projections = torch.nn.ModuleDict({
                name: projection_mlp(
                    CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers
                )
                for name in CARD_REGION_NAMES
            })
        self.own_summary_projection = projection_mlp(
            OWN_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        self.opponent_summary_projection = projection_mlp(
            OPPONENT_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        self.global_summary_projection = projection_mlp(
            GLOBAL_SUMMARY_DIM, config.d_model, config.summary_mlp_layers
        )
        encoder_card_ids, encoder_card_regions = encoder_card_mappings(config)
        prenorm = config.norm_mode == 'prenorm'
        self.encoder_bag = CardAwareEmbeddingBag(
            config.encoder_size, config.d_model,
            encoder_card_ids, encoder_card_regions,
        )
        self.pokemon_appear_embedding = (
            torch.nn.Embedding(3, config.d_model, padding_idx=0)
            if config.pokemon_appear_embedding else None
        )
        self.own_bench_token_mlp = self.make_token_mlp(config.bench_token_mlp_layers)
        self.opponent_bench_token_mlp = self.make_token_mlp(config.bench_token_mlp_layers)
        self.own_active_token_mlp = self.make_token_mlp(config.active_token_mlp_layers)
        self.opponent_active_token_mlp = self.make_token_mlp(config.active_token_mlp_layers)
        self.own_discard_token_mlp = self.make_token_mlp(config.discard_token_mlp_layers)
        self.opponent_discard_token_mlp = self.make_token_mlp(config.discard_token_mlp_layers)
        self.own_hand_token_mlp = self.make_token_mlp(config.hand_token_mlp_layers)
        self.own_deck_token_mlp = self.make_token_mlp(config.deck_token_mlp_layers)
        self.own_revealed_hand_token_mlp = self.make_token_mlp(
            config.revealed_hand_token_mlp_layers
        )
        self.opponent_revealed_hand_token_mlp = self.make_token_mlp(
            config.revealed_hand_token_mlp_layers
        )
        self.cls_token = (torch.nn.Parameter(torch.empty(1, 1, config.d_model))
                          if config.learnable_cls_token else None)
        if self.cls_token is not None:
            torch.nn.init.normal_(self.cls_token, mean=0.0, std=0.02)
        self.register_buffer(
            'own_area_card_regions', torch.tensor(OWN_AREA_REGIONS), persistent=False
        )
        self.register_buffer(
            'opponent_area_card_regions',
            torch.tensor(OPPONENT_AREA_REGIONS), persistent=False,
        )
        layer = EncoderLayer(
            config.d_model, config.num_heads, config.d_feedforward, config.norm_mode,
            config.transformer_activation, config.transformer_dropout,
            config.dropout_attention_probs, config.dropout_attention_output,
            config.dropout_ffn_output,
        )
        self.encoder = torch.nn.TransformerEncoder(
            layer, config.encoder_layers,
            norm=torch.nn.LayerNorm(config.d_model) if prenorm else None,
            enable_nested_tensor=False,
        )
        self.encoder_input_norm = (
            torch.nn.LayerNorm(config.d_model) if config.dropout_embedding else None
        )
        self.action_input_norm = (
            torch.nn.LayerNorm(config.d_model) if config.dropout_embedding else None
        )
        self.embedding_dropout = torch.nn.Dropout(
            config.transformer_dropout if config.dropout_embedding else 0.0
        )
        self.option_type_embedding = torch.nn.Embedding(OPTION_TYPE_COUNT, config.d_model)
        self.option_context_embedding = torch.nn.Embedding(OPTION_CONTEXT_COUNT, config.d_model)
        self.option_candidate_embedding = torch.nn.Embedding(
            config.card_count + 1, config.d_model, padding_idx=config.card_count
        )
        self.option_target_embedding = torch.nn.Embedding(
            config.card_count + 1, config.d_model, padding_idx=config.card_count
        )
        self.option_attack_embedding = torch.nn.Embedding(
            config.attack_count + 1, config.d_model, padding_idx=config.attack_count
        )
        self.option_number_embedding = torch.nn.Embedding(
            OPTION_VALUE_COUNT, config.d_model, padding_idx=0
        )
        self.option_count_embedding = torch.nn.Embedding(
            OPTION_VALUE_COUNT, config.d_model, padding_idx=0
        )
        self.option_player_relation_embedding = torch.nn.Embedding(
            OPTION_PLAYER_RELATION_COUNT, config.d_model, padding_idx=0
        )
        self.option_area_embedding = torch.nn.Embedding(
            OPTION_AREA_COUNT, config.d_model, padding_idx=0
        )
        self.option_in_play_area_embedding = torch.nn.Embedding(
            OPTION_AREA_COUNT, config.d_model, padding_idx=0
        )
        self.option_special_condition_embedding = torch.nn.Embedding(
            OPTION_SPECIAL_CONDITION_COUNT, config.d_model, padding_idx=0
        )
        self.option_numeric_projection = projection_mlp(
            OPTION_NUMERIC_DIM, config.d_model, config.option_numeric_mlp_layers
        )
        self.pokemon_dynamic_projection = torch.nn.Linear(
            POKEMON_DYNAMIC_DIM, config.d_model
        )
        self.attack_dynamic_projection = torch.nn.Linear(
            ATTACK_DYNAMIC_DIM, config.d_model
        )
        self.option_token_mlp = self.make_token_mlp(
            config.option_token_mlp_layers
        )
        self.attack_feature_projection = torch.nn.Linear(ATTACK_FEATURE_DIM, config.d_model)
        self.no_action_embedding = torch.nn.Parameter(torch.zeros(config.d_model))
        self.history_select_type_embedding = None
        self.history_context_embedding = None
        self.history_no_action_embedding = None
        self.history_action_mlp = None
        self.history_sequence_mlp = None
        if config.history_encoding != 'off':
            self.history_select_type_embedding = torch.nn.Embedding(SELECT_TYPE_DIM, config.d_model)
            self.history_context_embedding = torch.nn.Embedding(OPTION_CONTEXT_COUNT, config.d_model)
            self.history_no_action_embedding = torch.nn.Parameter(torch.zeros(config.d_model))
            self.history_option_type_embedding = torch.nn.Embedding(OPTION_TYPE_COUNT, config.d_model)
            if config.history_encoding == 'structural':
                self.history_source_area_embedding = torch.nn.Embedding(OPTION_AREA_COUNT, config.d_model, padding_idx=0)
                self.history_target_area_embedding = torch.nn.Embedding(OPTION_AREA_COUNT, config.d_model, padding_idx=0)
                self.history_source_relation_embedding = torch.nn.Embedding(OPTION_PLAYER_RELATION_COUNT, config.d_model, padding_idx=0)
                self.history_target_relation_embedding = torch.nn.Embedding(OPTION_PLAYER_RELATION_COUNT, config.d_model, padding_idx=0)
                self.history_number_embedding = torch.nn.Embedding(OPTION_VALUE_COUNT, config.d_model, padding_idx=0)
                self.history_count_embedding = torch.nn.Embedding(OPTION_VALUE_COUNT, config.d_model, padding_idx=0)
                self.history_special_condition_embedding = torch.nn.Embedding(OPTION_SPECIAL_CONDITION_COUNT, config.d_model, padding_idx=0)
            elif config.history_encoding == 'full':
                self.history_candidate_embedding = torch.nn.Embedding(config.card_count + 1, config.d_model, padding_idx=config.card_count)
                self.history_target_embedding = torch.nn.Embedding(config.card_count + 1, config.d_model, padding_idx=config.card_count)
                self.history_attack_embedding = torch.nn.Embedding(config.attack_count + 1, config.d_model, padding_idx=config.attack_count)
                self.history_number_embedding = torch.nn.Embedding(OPTION_VALUE_COUNT, config.d_model, padding_idx=0)
                self.history_count_embedding = torch.nn.Embedding(OPTION_VALUE_COUNT, config.d_model, padding_idx=0)
                self.history_player_relation_embedding = torch.nn.Embedding(OPTION_PLAYER_RELATION_COUNT, config.d_model, padding_idx=0)
                self.history_area_embedding = torch.nn.Embedding(OPTION_AREA_COUNT, config.d_model, padding_idx=0)
                self.history_in_play_area_embedding = torch.nn.Embedding(OPTION_AREA_COUNT, config.d_model, padding_idx=0)
                self.history_special_condition_embedding = torch.nn.Embedding(OPTION_SPECIAL_CONDITION_COUNT, config.d_model, padding_idx=0)
                self.history_candidate_static_projection = None
                self.history_target_static_projection = None
                if config.card_mlp_layers > 0:
                    self.history_candidate_static_projection = projection_mlp(CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers)
                    self.history_target_static_projection = projection_mlp(CARD_FEATURE_DIM, config.d_model, config.card_mlp_layers)
                self.history_attack_static_projection = torch.nn.Linear(ATTACK_FEATURE_DIM, config.d_model)
                self.history_pokemon_dynamic_projection = torch.nn.Linear(POKEMON_DYNAMIC_DIM, config.d_model)
                self.history_attack_dynamic_projection = torch.nn.Linear(ATTACK_DYNAMIC_DIM, config.d_model)
            self.history_action_mlp = self.make_token_mlp(config.history_action_mlp_layers)
            self.history_sequence_mlp = projection_mlp(HISTORY_STEPS * config.d_model, config.d_model, config.history_sequence_mlp_layers)
        self.decoder = torch.nn.ModuleList(
            DecoderLayer(
                config.d_model, config.num_heads,
                config.d_feedforward, config.norm_mode,
                config.transformer_activation, config.transformer_dropout,
                config.dropout_attention_probs, config.dropout_attention_output,
                config.dropout_ffn_output,
            )
            for _ in range(config.decoder_layers)
        )
        self.decoder_fc = torch.nn.Linear(config.d_model, 1)

    def make_token_mlp(self, layers):
        if layers == 0:
            return None
        return projection_mlp(self.config.d_model, self.config.d_model, layers)

    def apply_token_mlp(self, tokens, mlp):
        if mlp is None:
            return tokens
        transformed = mlp(tokens)
        if self.config.region_token_mlp_residual:
            return tokens + transformed
        return transformed

    def apply_region_token_mlps(self, encoded):
        return torch.cat((
            self.apply_token_mlp(encoded[:, 0:8], self.own_bench_token_mlp),
            self.apply_token_mlp(encoded[:, 8:16], self.opponent_bench_token_mlp),
            self.apply_token_mlp(encoded[:, 16:17], self.own_active_token_mlp),
            self.apply_token_mlp(encoded[:, 17:18], self.opponent_active_token_mlp),
            encoded[:, 18:20],
            self.apply_token_mlp(encoded[:, 20:21], self.own_discard_token_mlp),
            self.apply_token_mlp(encoded[:, 21:22], self.opponent_discard_token_mlp),
            self.apply_token_mlp(encoded[:, 22:23], self.own_hand_token_mlp),
            self.apply_token_mlp(encoded[:, 23:24], self.own_deck_token_mlp),
            encoded[:, 24:26],
            self.apply_token_mlp(
                encoded[:, 26:27], self.own_revealed_hand_token_mlp
            ),
            self.apply_token_mlp(
                encoded[:, 27:28], self.opponent_revealed_hand_token_mlp
            ),
        ), dim=1)

    def project_card_features(self):
        if (
            self.card_feature_projection is None
            and self.card_feature_projections is None
        ):
            projected = self.card_feature_table.new_zeros((
                self.config.card_count, self.config.d_model
            ))
        elif self.card_feature_projection is not None:
            projected = self.card_feature_projection(self.card_feature_table)
        else:
            projected = torch.stack([
                self.card_feature_projections[name](self.card_feature_table)
                for name in CARD_REGION_NAMES
            ])
        if projected.ndim == 2:
            return torch.cat([
                projected, projected.new_zeros((1, self.config.d_model))
            ])
        return torch.cat([
            projected,
            projected.new_zeros((projected.size(0), 1, self.config.d_model)),
        ], dim=1)

    def decoder_card_regions(self, categorical):
        option_types = categorical[:, 0]
        player_relation = categorical[:, OPTION_PLAYER_RELATION_INDEX]
        areas = categorical[:, OPTION_AREA_INDEX]
        in_play_areas = categorical[:, OPTION_IN_PLAY_AREA_INDEX]
        candidate_regions = self.own_area_card_regions[areas]
        candidate_regions = torch.where(
            player_relation == 2,
            self.opponent_area_card_regions[areas], candidate_regions,
        )
        candidate_regions = torch.where(
            option_types == OPTION_TYPE_PLAY,
            torch.full_like(candidate_regions, CARD_REGION_INDEX['own_hand']),
            candidate_regions,
        )
        target_regions = torch.full_like(
            candidate_regions, CARD_REGION_INDEX['unknown']
        )
        has_in_play_target = (
            (option_types == OPTION_TYPE_ATTACH)
            | (option_types == OPTION_TYPE_EVOLVE)
        )
        target_regions = torch.where(
            has_in_play_target, self.own_area_card_regions[in_play_areas],
            target_regions,
        )
        target_regions = torch.where(
            option_types == OPTION_TYPE_RETREAT,
            torch.full_like(target_regions, CARD_REGION_INDEX['own_active']),
            target_regions,
        )
        return candidate_regions, target_regions

    def project_attack_features(self):
        projected = self.attack_feature_projection(self.attack_feature_table)
        return torch.cat([
            projected, projected.new_zeros((1, self.config.d_model))
        ])

    def project_pokemon_dynamic(self, features):
        present = (
            (features[:, 0] > 0)
            | (features[:, POKEMON_DYNAMIC_WORD_DIM] > 0)
        )
        return self.pokemon_dynamic_projection(features) * present.unsqueeze(1)

    def project_attack_dynamic(self, features):
        present = features[:, 0] > 0
        return self.attack_dynamic_projection(features) * present.unsqueeze(1)

    def apply_option_token_mlp(self, token):
        return token if self.option_token_mlp is None else self.option_token_mlp(token)

    def encode_options(self, categorical, numeric, pokemon_dynamic, attack_dynamic,
                       card_static, attack_static):
        candidate_ids = categorical[:, 2]
        target_ids = categorical[:, 3]
        attack_ids = categorical[:, 4]
        if card_static.ndim == 3:
            candidate_regions, target_regions = self.decoder_card_regions(
                categorical
            )
            candidate_static = card_static[candidate_regions, candidate_ids]
            target_static = card_static[target_regions, target_ids]
        else:
            candidate_static = card_static[candidate_ids]
            target_static = card_static[target_ids]
        token = (
            self.option_type_embedding(categorical[:, 0])
            + self.option_context_embedding(categorical[:, 1])
            + self.option_candidate_embedding(candidate_ids)
            + self.option_target_embedding(target_ids)
            + self.option_attack_embedding(attack_ids)
            + self.option_number_embedding(categorical[:, 5])
            + self.option_count_embedding(categorical[:, 6])
            + self.option_player_relation_embedding(categorical[:, 7])
            + self.option_area_embedding(categorical[:, 8])
            + self.option_in_play_area_embedding(categorical[:, 9])
            + self.option_special_condition_embedding(categorical[:, 10])
            + self.option_numeric_projection(numeric)
            + candidate_static
            + target_static
            + attack_static[attack_ids]
            + self.project_pokemon_dynamic(pokemon_dynamic)
            + self.project_attack_dynamic(attack_dynamic)
        )
        return self.apply_option_token_mlp(token)

    def combine_actions(self, option_embeddings, action_index, action_offset):
        if action_index.numel() == 0:
            combined = option_embeddings.new_zeros((
                action_offset.numel() - 1, self.config.d_model
            ))
        else:
            combined = torch.nn.functional.embedding_bag(
                action_index, option_embeddings, action_offset,
                mode='sum', include_last_offset=True,
            )
        empty = action_offset[1:] == action_offset[:-1]
        return combined + empty.unsqueeze(1) * self.no_action_embedding

    def encode_history_option_rows(self, categorical, structural, pokemon_dynamic, attack_dynamic):
        mode = self.config.history_encoding
        token = self.history_option_type_embedding(categorical[:, 0])
        if mode == 'basic':
            return token
        if mode == 'structural':
            return (
                self.history_option_type_embedding(structural[:, HISTORY_OPTION_TYPE_INDEX])
                + self.history_source_area_embedding(structural[:, HISTORY_SOURCE_AREA_INDEX])
                + self.history_target_area_embedding(structural[:, HISTORY_TARGET_AREA_INDEX])
                + self.history_source_relation_embedding(structural[:, HISTORY_SOURCE_RELATION_INDEX])
                + self.history_target_relation_embedding(structural[:, HISTORY_TARGET_RELATION_INDEX])
                + self.history_number_embedding(structural[:, HISTORY_NUMBER_INDEX])
                + self.history_count_embedding(structural[:, HISTORY_COUNT_INDEX])
                + self.history_special_condition_embedding(structural[:, HISTORY_SPECIAL_CONDITION_INDEX])
            )
        candidate_ids, target_ids, attack_ids = categorical[:, 2], categorical[:, 3], categorical[:, 4]
        if self.history_candidate_static_projection is None:
            candidate_static = token.new_zeros(token.shape)
            target_static = token.new_zeros(token.shape)
        else:
            candidate_table = self.history_candidate_static_projection(self.card_feature_table)
            target_table = self.history_target_static_projection(self.card_feature_table)
            candidate_static = torch.cat((candidate_table, candidate_table.new_zeros((1, self.config.d_model))))[candidate_ids]
            target_static = torch.cat((target_table, target_table.new_zeros((1, self.config.d_model))))[target_ids]
        attack_table = self.history_attack_static_projection(self.attack_feature_table)
        attack_static = torch.cat((attack_table, attack_table.new_zeros((1, self.config.d_model))))[attack_ids]
        pokemon_present = (pokemon_dynamic[:, 0] > 0) | (pokemon_dynamic[:, POKEMON_DYNAMIC_WORD_DIM] > 0)
        pokemon_token = self.history_pokemon_dynamic_projection(pokemon_dynamic) * pokemon_present.unsqueeze(1)
        attack_present = attack_dynamic[:, 0] > 0
        attack_token = self.history_attack_dynamic_projection(attack_dynamic) * attack_present.unsqueeze(1)
        return (token + self.history_candidate_embedding(candidate_ids)
                + self.history_target_embedding(target_ids) + self.history_attack_embedding(attack_ids)
                + self.history_number_embedding(categorical[:, 5]) + self.history_count_embedding(categorical[:, 6])
                + self.history_player_relation_embedding(categorical[:, 7]) + self.history_area_embedding(categorical[:, 8])
                + self.history_in_play_area_embedding(categorical[:, 9]) + self.history_special_condition_embedding(categorical[:, 10])
                + candidate_static + target_static + attack_static + pokemon_token + attack_token)

    def encode_history(self, select_type, select_context, valid, categorical, structural, pokemon_dynamic, attack_dynamic, offsets):
        slots = valid.numel()
        option_tokens = self.encode_history_option_rows(categorical, structural, pokemon_dynamic, attack_dynamic)
        if option_tokens.size(0) == 0:
            actions = option_tokens.new_zeros((slots, self.config.d_model))
        else:
            option_indices = torch.arange(option_tokens.size(0), device=option_tokens.device)
            actions = torch.nn.functional.embedding_bag(option_indices, option_tokens, offsets, mode='sum', include_last_offset=True)
        flat_valid = valid.reshape(-1).bool()
        actions = actions + self.history_select_type_embedding(select_type.reshape(-1))
        actions = actions + self.history_context_embedding(select_context.reshape(-1))
        empty = offsets[1:] == offsets[:-1]
        actions = actions + (empty & flat_valid).unsqueeze(1) * self.history_no_action_embedding
        if self.history_action_mlp is not None:
            actions = self.history_action_mlp(actions)
        actions = actions * flat_valid.unsqueeze(1)
        actions = actions.reshape(valid.size(0), HISTORY_STEPS * self.config.d_model)
        return self.history_sequence_mlp(actions) * valid.any(dim=1, keepdim=True)

    def _encoder_padding_mask(
        self, own_summary, opponent_summary, revealed_hand_present,
        history_valid=None,
    ):
        slots = torch.arange(BENCH_SLOTS, device=own_summary.device)
        own_count = torch.round(
            own_summary[:, PLAYER_BENCH_COUNT_INDEX] * BENCH_SLOTS
        ).to(torch.long).clamp(0, BENCH_SLOTS)
        opponent_count = torch.round(
            opponent_summary[:, PLAYER_BENCH_COUNT_INDEX] * BENCH_SLOTS
        ).to(torch.long).clamp(0, BENCH_SLOTS)
        own_padding = slots.unsqueeze(0) >= own_count.unsqueeze(1)
        opponent_padding = slots.unsqueeze(0) >= opponent_count.unsqueeze(1)
        fixed_tokens = torch.zeros(
            (own_summary.size(0), 26 - 2 * BENCH_SLOTS),
            dtype=torch.bool, device=own_summary.device,
        )
        revealed_padding = ~revealed_hand_present.bool()
        result = torch.cat((
            own_padding, opponent_padding, fixed_tokens, revealed_padding
        ), dim=1)
        if history_valid is not None:
            result = torch.cat((result, ~history_valid.bool().any(dim=1, keepdim=True)), dim=1)
        if self.cls_token is not None:
            result = torch.cat((
                result, torch.zeros(
                    (result.size(0), 1), dtype=torch.bool, device=result.device
                )
            ), dim=1)
        return result

    def forward(self, index_encoder, value_encoder, offset_encoder,
                pokemon_appear, own_summary, opponent_summary, global_summary,
                revealed_hand_present,
                history_select_type, history_select_context, history_valid,
                history_option_categorical, history_structural,
                history_pokemon_dynamic, history_attack_dynamic, history_option_offset,
                option_categorical, option_numeric, pokemon_dynamic, attack_dynamic,
                action_option_index, action_option_offset):
        cfg = self.config
        projected_card_features = self.project_card_features()
        projected_attack_features = self.project_attack_features()
        encoded = self.encoder_bag(
            index_encoder, offset_encoder, value_encoder, projected_card_features
        )
        batch_size = own_summary.size(0)
        encoded = encoded.reshape(batch_size, ENCODER_TOKENS, cfg.d_model)
        if self.pokemon_appear_embedding is not None:
            pokemon_tokens = (
                encoded[:, :POKEMON_ENCODER_TOKENS]
                + self.pokemon_appear_embedding(pokemon_appear)
            )
            encoded = torch.cat((
                pokemon_tokens, encoded[:, POKEMON_ENCODER_TOKENS:]
            ), dim=1)
        encoded = torch.cat((
            encoded[:, :18], self.own_summary_projection(own_summary).unsqueeze(1),
            self.opponent_summary_projection(opponent_summary).unsqueeze(1),
            encoded[:, 20:25], self.global_summary_projection(global_summary).unsqueeze(1),
            encoded[:, 26:28],
        ), dim=1)
        encoded = self.apply_region_token_mlps(encoded)
        if cfg.history_encoding != 'off':
            history_token = self.encode_history(
                history_select_type, history_select_context, history_valid,
                history_option_categorical, history_structural,
                history_pokemon_dynamic, history_attack_dynamic, history_option_offset,
            )
            encoded = torch.cat((encoded, history_token.unsqueeze(1)), dim=1)
        if self.cls_token is not None:
            encoded = torch.cat((
                encoded, self.cls_token.expand(batch_size, -1, -1)
            ), dim=1)
        if self.encoder_input_norm is not None:
            encoded = self.embedding_dropout(self.encoder_input_norm(encoded))
        encoded = encoded.transpose(0, 1)
        encoder_padding_mask = self._encoder_padding_mask(
            own_summary, opponent_summary, revealed_hand_present,
            history_valid if cfg.history_encoding != 'off' else None,
        )
        encoder_out = self.encoder(
            encoded, src_key_padding_mask=encoder_padding_mask
        )
        option_embeddings = self.encode_options(
            option_categorical, option_numeric, pokemon_dynamic, attack_dynamic,
            projected_card_features, projected_attack_features,
        )
        policy = self.combine_actions(
            option_embeddings, action_option_index, action_option_offset
        )
        if self.action_input_norm is not None:
            policy = self.embedding_dropout(self.action_input_norm(policy))
        policy = policy.reshape(batch_size, -1, cfg.d_model).transpose(0, 1)
        for layer in self.decoder:
            policy = layer(policy, encoder_out, encoder_padding_mask)
        return self.decoder_fc(policy).transpose(0, 1).reshape(batch_size, -1)


@dataclass
class SparseVector:
    index: list[int] = field(default_factory=list)
    value: list[float] = field(default_factory=list)
    offset: list[int] = field(default_factory=list)
    pos: int = 0

    def add(self, index, value):
        if float(value) != 0.0:
            self.index.append(self.pos + int(index))
            self.value.append(float(value))

    def add_pos(self, count):
        self.pos += count

    def add_single(self, value):
        self.add(0, value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))


def enumerate_actions(option_count, min_count, max_count):
    actions = []
    for count in range(max_count, min_count - 1, -1):
        for selection in combinations(range(option_count), count):
            actions.append(list(selection))
            if len(actions) == MAX_ACTIONS:
                return actions
    return actions


def damage_counter_action_eligibility(obs, actions):
    result = np.ones(len(actions), dtype=np.bool_)
    if int(obs.select.context) != 14:
        return result

    options = list(obs.select.option)
    option_eligible = np.ones(len(options), dtype=np.bool_)
    players = list(getattr(obs.current, 'players', ()) or ())
    for option_index, option in enumerate(options):
        if getattr(option, 'area', None) != AreaType.BENCH:
            continue
        raw_player_index = getattr(option, 'playerIndex', None)
        raw_bench_index = getattr(option, 'index', None)
        player_index = -1 if raw_player_index is None else int(raw_player_index)
        bench_index = -1 if raw_bench_index is None else int(raw_bench_index)
        if not 0 <= player_index < len(players):
            continue
        bench = list(getattr(players[player_index], 'bench', ()) or ())
        if not 0 <= bench_index < len(bench):
            continue
        hp = getattr(bench[bench_index], 'hp', None)
        if hp is not None and float(hp) <= 0:
            option_eligible[option_index] = False

    for action_index, action in enumerate(actions):
        result[action_index] = all(
            0 <= option_index < len(options)
            and bool(option_eligible[option_index])
            for option_index in action
        )
    if len(result) and not bool(result.any()):
        result.fill(True)
    return result


def add_card(sv, card, card_count):
    if card is not None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)


def add_cards(sv, cards, weight, card_count):
    if cards is not None:
        for card in cards:
            sv.add(card.id, weight)
    sv.add_pos(card_count)


def add_card_ids(sv, card_ids, weight, card_count):
    for card_id in card_ids:
        card_id = int(card_id)
        if not 0 <= card_id < card_count:
            raise ValueError(f'revealed card ID {card_id} is out of range')
        sv.add(card_id, weight)
    sv.add_pos(card_count)


def add_pokemon(sv, pokemon, card_count):
    if pokemon is None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
        return
    sv.add_single(0)
    sv.add_single(pokemon.hp / 400)
    add_card(sv, pokemon, card_count)
    add_cards(sv, pokemon.tools, 1, card_count)
    add_cards(sv, pokemon.energyCards, 0.5, card_count)


def build_numeric_catalog(cards, attacks, card_count):
    card_features = build_card_feature_table(cards, card_count)
    attack_count = max((int(attack.attackId) for attack in attacks), default=-1) + 1
    attack_damage = torch.zeros(attack_count, dtype=torch.float32)
    for attack in attacks:
        attack_damage[int(attack.attackId)] = float(attack.damage) / 300.0
    card_attacks = [()] * card_count
    for card in cards:
        if 0 <= int(card.cardId) < card_count:
            card_attacks[int(card.cardId)] = tuple(int(value) for value in card.attacks)
    return card_features, attack_damage, tuple(card_attacks)


def one_hot(index, size, name):
    index = int(index)
    if not 0 <= index < size:
        raise ValueError(f'{name}={index} is outside [0, {size})')
    result = [0.0] * size
    result[index] = 1.0
    return result


def active_card(player):
    return player.active[0] if player.active else None


def player_summary(player, card_features, attack_damage, card_attacks):
    active = active_card(player)
    bench = list(player.bench[:8])
    bench_energy = sum(len(pokemon.energyCards) for pokemon in bench)
    bench_hp = sum(float(pokemon.hp) for pokemon in bench)
    bench_max_hp = sum(float(pokemon.maxHp) for pokemon in bench)
    retreat_cost = max_attack_damage = attack_count = 0.0
    weakness_norm = resistance_norm = 0.0
    if active is not None and 0 <= int(active.id) < len(card_features):
        row = card_features[int(active.id)]
        retreat_cost = float(row[CARD_RETREAT_INDEX])
        weakness_norm = float(torch.argmax(row[CARD_WEAKNESS_OFFSET:CARD_WEAKNESS_OFFSET + CARD_WEAKNESS_DIM])) / 12.0
        resistance_norm = float(torch.argmax(row[CARD_RESISTANCE_OFFSET:CARD_RESISTANCE_OFFSET + CARD_RESISTANCE_DIM])) / 12.0
        attacks = card_attacks[int(active.id)]
        attack_count = len(attacks) / 4.0
        max_attack_damage = max((float(attack_damage[a]) for a in attacks if 0 <= a < len(attack_damage)), default=0.0)
    result = [player.deckCount / 60.0, player.handCount / 20.0, len(player.discard) / 60.0]
    result.extend(one_hot(len(player.prize), 7, 'prize_count'))
    result.extend([
        len(bench) / 8.0, float(active is not None),
        float(bool(player.poisoned)), float(bool(player.burned)),
        float(bool(player.asleep)), float(bool(player.paralyzed)), float(bool(player.confused)),
        float(active.hp) / 400.0 if active is not None else 0.0,
        float(active.maxHp) / 400.0 if active is not None else 0.0,
        len(active.energyCards) / 10.0 if active is not None else 0.0,
        len(active.tools) / 4.0 if active is not None else 0.0,
        len(active.preEvolution) / 2.0 if active is not None else 0.0,
        retreat_cost, max_attack_damage, attack_count, weakness_norm, resistance_norm,
    ])
    for slot in range(8):
        if slot < len(bench):
            pokemon = bench[slot]
            result.extend([1.0, float(pokemon.hp) / 400.0, len(pokemon.energyCards) / 5.0])
        else:
            result.extend([0.0, 0.0, 0.0])
    result.extend([bench_energy / 32.0, bench_hp / 3200.0, bench_max_hp / 3200.0])
    return result


def visible_own_cards(player):
    visible = Counter()
    def add(card):
        if card is not None:
            visible[int(card.id)] += 1
    def add_pokemon(pokemon):
        if pokemon is None:
            return
        add(pokemon)
        for attached in (pokemon.energyCards, pokemon.tools, pokemon.preEvolution):
            for card in attached:
                add(card)
    for card in player.hand or []:
        add(card)
    for card in player.discard:
        add(card)
    add_pokemon(active_card(player))
    for pokemon in player.bench:
        add_pokemon(pokemon)
    return visible


def deck_remaining_summary(deck, player, card_features):
    remaining = Counter(map(int, deck))
    remaining.subtract(visible_own_cards(player))
    remaining = Counter({card_id: max(0, count) for card_id, count in remaining.items()})
    total = float(sum(remaining.values()))
    type_counts = [0.0] * CARD_TYPE_DIM
    stage_counts = [0.0, 0.0, 0.0]
    has_ex = has_mega = 0.0
    for card_id, count in remaining.items():
        if count <= 0 or not 0 <= card_id < len(card_features):
            continue
        row = card_features[card_id]
        card_type = int(torch.argmax(row[:CARD_TYPE_DIM]))
        type_counts[card_type] += count
        for stage in range(3):
            if row[CARD_STAGE_OFFSET + stage] > 0.5:
                stage_counts[stage] += count
        has_ex = max(has_ex, float(row[CARD_SPECIAL_OFFSET] > 0.5))
        has_mega = max(has_mega, float(row[CARD_SPECIAL_OFFSET + 1] > 0.5))
    pokemon, energy = type_counts[0], type_counts[5] + type_counts[6]
    return ([value / 4.0 for value in type_counts]
            + [value / 4.0 for value in stage_counts] + [has_ex, has_mega]
            + [total / 60.0, pokemon / max(total, 1.0), energy / max(total, 1.0)])


def opponent_revealed_summary(player, card_features):
    revealed = list(player.discard)
    def add_pokemon(pokemon):
        if pokemon is not None:
            revealed.append(pokemon)
            revealed.extend(pokemon.energyCards)
            revealed.extend(pokemon.tools)
            revealed.extend(pokemon.preEvolution)
    add_pokemon(active_card(player))
    for pokemon in player.bench:
        add_pokemon(pokemon)
    card_ids = [int(card.id) for card in revealed if 0 <= int(card.id) < len(card_features)]
    type_counts = [0.0] * CARD_TYPE_DIM
    has_ex = has_mega = has_tera = 0.0
    pokemon_count = 0
    average_hp = average_stage = 0.0
    for card_id in card_ids:
        row = card_features[card_id]
        card_type = int(torch.argmax(row[:CARD_TYPE_DIM]))
        type_counts[card_type] += 1.0
        if card_type == 0:
            pokemon_count += 1
            average_hp += float(row[CARD_HP_INDEX])
            has_ex = max(has_ex, float(row[CARD_SPECIAL_OFFSET] > 0.5))
            has_mega = max(has_mega, float(row[CARD_SPECIAL_OFFSET + 1] > 0.5))
            has_tera = max(has_tera, float(row[CARD_SPECIAL_OFFSET + 2] > 0.5))
            average_stage += float(row[CARD_STAGE_OFFSET + 1]) + 2.0 * float(row[CARD_STAGE_OFFSET + 2])
    if pokemon_count:
        average_hp /= pokemon_count
        average_stage /= pokemon_count
    energy_in_play = sum(len(p.energyCards) for p in ([active_card(player)] + list(player.bench)) if p is not None)
    return ([value / 10.0 for value in type_counts] + [has_ex, has_mega, has_tera]
            + [pokemon_count / 5.0, len(card_ids) / 20.0, average_hp,
               average_stage / 2.0, len(player.discard) / 20.0,
               len(player.bench) / 8.0, energy_in_play / 10.0])


def global_summary(obs, yours):
    state, select = obs.current, obs.select
    first_relative = -1.0 if int(state.firstPlayer) < 0 else float(int(state.firstPlayer) == yours)
    result = [state.turn / 100.0, state.turnActionCount / 100.0, first_relative,
              float(bool(state.supporterPlayed)), float(bool(state.stadiumPlayed)),
              float(bool(state.energyAttached)), float(bool(state.retreated)), float(yours)]
    result.extend(one_hot(int(select.type), SELECT_TYPE_DIM, 'select.type'))
    result.extend(one_hot(int(select.context), SELECT_CONTEXT_DIM, 'select.context'))
    result.extend([select.minCount / 6.0, select.maxCount / 6.0,
                   select.remainDamageCounter / 20.0, select.remainEnergyCost / 10.0,
                   len(select.option) / 64.0])
    return result


def encoder_features(
    obs, deck, card_count, catalog, own_revealed=(), opponent_revealed=()
):
    card_features, attack_damage, card_attacks = catalog
    state, yours, sv = obs.current, obs.current.yourIndex, SparseVector()
    players = [state.players[yours], state.players[1 - yours]]
    pokemon_appear = []
    for player in players:
        for slot in range(8):
            pokemon = player.bench[slot] if slot < len(player.bench) else None
            pokemon_appear.append(
                0 if pokemon is None else 2 if bool(pokemon.appearThisTurn) else 1
            )
            sv.word_start()
            pos = sv.pos
            add_pokemon(sv, pokemon, card_count)
            if slot != 7:
                sv.pos = pos
    for player in players:
        pokemon = active_card(player)
        pokemon_appear.append(
            0 if pokemon is None else 2 if bool(pokemon.appearThisTurn) else 1
        )
        sv.word_start()
        add_pokemon(sv, pokemon, card_count)
    sv.word_start()
    sv.word_start()
    sv.word_start()
    add_cards(sv, players[0].discard, 0.25, card_count)
    sv.word_start()
    add_cards(sv, players[1].discard, 0.25, card_count)
    sv.word_start()
    add_cards(sv, players[0].hand, 0.25, card_count)
    sv.word_start()
    for card_id in deck:
        sv.add(card_id, 0.25)
    sv.add_pos(card_count)
    sv.word_start()
    add_cards(sv, state.stadium, 1.0, card_count)
    sv.word_start()
    add_card_ids(sv, own_revealed, 1.0, card_count)
    sv.word_start()
    add_card_ids(sv, opponent_revealed, 1.0, card_count)
    own = player_summary(players[0], card_features, attack_damage, card_attacks)
    own.extend(deck_remaining_summary(deck, players[0], card_features))
    opponent = player_summary(players[1], card_features, attack_damage, card_attacks)
    opponent.extend(opponent_revealed_summary(players[1], card_features))
    return (sv, pokemon_appear, own, opponent, global_summary(obs, yours),
            [int(bool(own_revealed)), int(bool(opponent_revealed))])


def optional_int(value, default=0):
    return default if value is None else int(value)


def option_value_index(value, name):
    if value is None:
        return 0
    number = int(value)
    if number < 0:
        raise ValueError(f'{name}={number} must be non-negative')
    return 62 if number > 60 else number + 1


def area_card(obs, area, index, player_index):
    player = obs.current.players[player_index]
    mapping = {
        AreaType.DECK: obs.select.deck, AreaType.HAND: player.hand,
        AreaType.DISCARD: player.discard, AreaType.ACTIVE: player.active,
        AreaType.BENCH: player.bench, AreaType.PRIZE: player.prize,
        AreaType.STADIUM: obs.current.stadium, AreaType.LOOKING: obs.current.looking,
    }
    cards = list(mapping.get(area) or [])
    position = optional_int(index, -1)
    return cards[position] if 0 <= position < len(cards) else None


def valid_card_id(card, card_count):
    if card is None:
        return card_count
    card_id = int(card.id)
    return card_id if 0 <= card_id < card_count else card_count


def is_pokemon(value):
    return value is not None and all(
        hasattr(value, name)
        for name in ('hp', 'maxHp', 'energies', 'energyCards', 'tools')
    )


def pokemon_dynamic_features(pokemon, is_active, is_own):
    features = torch.zeros(POKEMON_DYNAMIC_WORD_DIM, dtype=torch.float32)
    if not is_pokemon(pokemon):
        return features
    hp = max(float(pokemon.hp), 0.0)
    max_hp = max(float(pokemon.maxHp), 0.0)
    tools = list(pokemon.tools or [])
    energy_cards = list(pokemon.energyCards or [])
    energies = list(pokemon.energies or [])
    features[:8] = torch.tensor([
        1.0, hp / 400.0, max_hp / 400.0,
        max(max_hp - hp, 0.0) / 400.0,
        hp / max_hp if max_hp > 0 else 0.0,
        len(tools) / 4.0, len(energy_cards) / 10.0, len(energies) / 10.0,
    ])
    for energy in energies:
        energy_type = int(energy)
        if 0 <= energy_type < ENERGY_TYPE_DIM:
            features[8 + energy_type] += 0.1
    features[20:] = torch.tensor([
        float(bool(getattr(pokemon, 'appearThisTurn', False))),
        float(is_active), float(is_own),
    ])
    return features


def option_pokemon_slots(obs, option):
    yours = int(obs.current.yourIndex)
    missing = (None, False, False)
    if option.type == OptionType.ATTACK:
        return (
            (active_card(obs.current.players[yours]), True, True),
            (active_card(obs.current.players[1 - yours]), True, False),
        )
    if option.type == OptionType.RETREAT:
        return ((active_card(obs.current.players[yours]), True, True), missing)
    player_index = optional_int(option.playerIndex, yours)
    player_index = max(0, min(player_index, len(obs.current.players) - 1))
    if option.type in {
        OptionType.ABILITY, OptionType.CARD, OptionType.DISCARD,
        OptionType.TOOL_CARD, OptionType.ENERGY_CARD, OptionType.ENERGY,
    }:
        pokemon = area_card(obs, option.area, option.index, player_index)
        return ((pokemon if is_pokemon(pokemon) else None,
                 option.area == AreaType.ACTIVE, player_index == yours), missing)
    if option.type in {OptionType.ATTACH, OptionType.EVOLVE}:
        pokemon = area_card(obs, option.inPlayArea, option.inPlayIndex, yours)
        return ((pokemon if is_pokemon(pokemon) else None,
                 option.inPlayArea == AreaType.ACTIVE, True), missing)
    return missing, missing

def option_entity_ids(obs, option, card_count, attack_count):
    yours = int(obs.current.yourIndex)
    player_index = optional_int(option.playerIndex, yours)
    player_index = max(0, min(player_index, len(obs.current.players) - 1))
    candidate = target = None
    if option.type == OptionType.PLAY:
        candidate = area_card(obs, AreaType.HAND, option.index, yours)
    elif option.type in {OptionType.CARD, OptionType.TOOL_CARD,
                          OptionType.ENERGY_CARD, OptionType.ENERGY,
                          OptionType.ABILITY, OptionType.DISCARD}:
        candidate = area_card(obs, option.area, option.index, player_index)
        if option.type == OptionType.TOOL_CARD and candidate is not None:
            cards = list(candidate.tools or [])
            index = optional_int(option.toolIndex, -1)
            candidate = cards[index] if 0 <= index < len(cards) else None
        elif option.type in {OptionType.ENERGY_CARD, OptionType.ENERGY} and candidate is not None:
            cards = list(candidate.energyCards or [])
            index = optional_int(option.energyIndex, -1)
            candidate = cards[index] if 0 <= index < len(cards) else None
    elif option.type in {OptionType.ATTACH, OptionType.EVOLVE}:
        candidate = area_card(obs, option.area, option.index, player_index)
        target = area_card(obs, option.inPlayArea, option.inPlayIndex, yours)
    elif option.type == OptionType.RETREAT:
        active = list(obs.current.players[yours].active or [])
        target = active[0] if active else None
    candidate_id = valid_card_id(candidate, card_count)
    if candidate_id == card_count:
        raw_card_id = optional_int(option.cardId, 0)
        if 0 < raw_card_id < card_count:
            candidate_id = raw_card_id
    target_id = valid_card_id(target, card_count)
    raw_attack_id = option.attackId
    attack_id = (int(raw_attack_id) if raw_attack_id is not None
                 and 0 <= int(raw_attack_id) < attack_count else attack_count)
    return candidate_id, target_id, attack_id


def decoder_features(obs, actions, card_count, attack_count, catalog):
    card_features, attack_damage, _ = catalog
    options = list(obs.select.option)
    categorical = torch.empty((len(options), 11), dtype=torch.int64)
    numeric = torch.zeros((len(options), OPTION_NUMERIC_DIM))
    pokemon_dynamic = torch.zeros((len(options), POKEMON_DYNAMIC_DIM))
    attack_dynamic = torch.zeros((len(options), ATTACK_DYNAMIC_DIM))
    yours = int(obs.current.yourIndex)
    context = int(obs.select.context)
    for position, option in enumerate(options):
        candidate_id, target_id, attack_id = option_entity_ids(
            obs, option, card_count, attack_count
        )
        player_relation = (0 if option.playerIndex is None else
                           1 if int(option.playerIndex) == yours else 2)
        area_index = 0 if option.area is None else int(option.area)
        in_play_area_index = (0 if option.inPlayArea is None
                              else int(option.inPlayArea))
        special_condition = (0 if option.specialConditionType is None
                             else int(option.specialConditionType) + 1)
        categorical[position] = torch.tensor([
            int(option.type), context, candidate_id, target_id, attack_id,
            option_value_index(option.number, 'option number'),
            option_value_index(option.count, 'option count'),
            player_relation, area_index, in_play_area_index, special_condition,
        ])
        numeric[position] = torch.tensor([
            optional_int(option.index) / 60.0,
            optional_int(option.toolIndex) / 4.0,
            optional_int(option.energyIndex) / 10.0,
            optional_int(option.inPlayIndex) / 5.0,
            (position + 1) / max(1, len(options)),
        ])
        primary, secondary = option_pokemon_slots(obs, option)
        primary_features = pokemon_dynamic_features(primary[0], primary[1], primary[2])
        secondary_features = pokemon_dynamic_features(secondary[0], secondary[1], secondary[2])
        pokemon_dynamic[position] = torch.cat((primary_features, secondary_features))
        if (option.type == OptionType.ATTACK and attack_id < attack_count
                and primary_features[0] > 0 and secondary_features[0] > 0):
            target_hp = max(float(secondary[0].hp), 0.0)
            base_damage = (float(attack_damage[attack_id]) * 300.0
                           if attack_id < len(attack_damage) else 0.0)
            super_effective = resisted = False
            own_id = valid_card_id(primary[0], card_count)
            opponent_id = valid_card_id(secondary[0], card_count)
            if own_id < card_count and opponent_id < card_count:
                own_energy = card_features[
                    own_id, CARD_ENERGY_TYPE_OFFSET:CARD_ENERGY_TYPE_OFFSET + ENERGY_TYPE_DIM
                ]
                if bool(torch.any(own_energy > 0.5)):
                    own_energy_type = int(torch.argmax(own_energy))
                    super_effective = bool(card_features[
                        opponent_id, CARD_WEAKNESS_OFFSET + own_energy_type
                    ] > 0.5)
                    resisted = bool(card_features[
                        opponent_id, CARD_RESISTANCE_OFFSET + own_energy_type
                    ] > 0.5)
            attack_dynamic[position] = torch.tensor([
                1.0, min(base_damage / max(target_hp, 1.0), 4.0) / 4.0,
                float(base_damage >= target_hp),
                max(target_hp - base_damage, 0.0) / 400.0,
                float(super_effective), float(resisted),
            ])
    action_index = []
    action_offset = [0]
    for action in actions:
        action_index.extend(action)
        action_offset.append(len(action_index))
    return (categorical, numeric, pokemon_dynamic, attack_dynamic,
            torch.tensor(action_index, dtype=torch.int64),
            torch.tensor(action_offset, dtype=torch.int64))


def history_structural_option(obs, option):
    yours = int(obs.current.yourIndex)
    source_area = 0 if option.area is None else int(option.area)
    target_area = 0 if option.inPlayArea is None else int(option.inPlayArea)
    source_relation = (0 if option.playerIndex is None else 1 if int(option.playerIndex) == yours else 2)
    target_relation = 0
    if option.type == OptionType.PLAY:
        source_area, source_relation = int(AreaType.HAND), 1
    elif option.type == OptionType.ATTACK:
        source_area = target_area = int(AreaType.ACTIVE)
        source_relation, target_relation = 1, 2
    elif option.type == OptionType.RETREAT:
        source_area, source_relation = int(AreaType.ACTIVE), 1
    elif option.type in {OptionType.ATTACH, OptionType.EVOLVE}:
        source_relation = source_relation or 1
        target_relation = int(target_area != 0)
    return [int(option.type), source_area, target_area, source_relation, target_relation,
            option_value_index(option.number, 'history option number'),
            option_value_index(option.count, 'history option count'),
            0 if option.specialConditionType is None else int(option.specialConditionType) + 1]


def completed_history_action(obs, selected, decoded):
    categorical, _, pokemon_dynamic, attack_dynamic, _, _ = decoded
    rows = torch.tensor(selected, dtype=torch.long)
    structural = torch.tensor(
        [history_structural_option(obs, obs.select.option[index]) for index in selected],
        dtype=torch.long,
    ).reshape(-1, HISTORY_STRUCTURAL_DIM)
    return {
        'select_type': int(obs.select.type),
        'select_context': int(obs.select.context),
        'categorical': categorical[rows].clone(),
        'structural': structural,
        'pokemon_dynamic': pokemon_dynamic[rows].clone(),
        'attack_dynamic': attack_dynamic[rows].clone(),
    }


def history_tensors(history):
    recent = list(history)[-HISTORY_STEPS:]
    slots = [None] * (HISTORY_STEPS - len(recent)) + recent
    select_type = torch.zeros((1, HISTORY_STEPS), dtype=torch.long)
    select_context = torch.zeros((1, HISTORY_STEPS), dtype=torch.long)
    valid = torch.zeros((1, HISTORY_STEPS), dtype=torch.long)
    categorical, structural, pokemon_dynamic, attack_dynamic = [], [], [], []
    offsets = [0]
    for slot_index, action in enumerate(slots):
        if action is not None:
            select_type[0, slot_index] = action['select_type']
            select_context[0, slot_index] = action['select_context']
            valid[0, slot_index] = 1
            categorical.append(action['categorical'])
            structural.append(action['structural'])
            pokemon_dynamic.append(action['pokemon_dynamic'])
            attack_dynamic.append(action['attack_dynamic'])
            offsets.append(offsets[-1] + action['categorical'].size(0))
        else:
            offsets.append(offsets[-1])
    def rows_or_empty(parts, width, dtype):
        return torch.cat(parts) if parts else torch.empty((0, width), dtype=dtype)
    return (select_type, select_context, valid,
            rows_or_empty(categorical, 11, torch.long),
            rows_or_empty(structural, HISTORY_STRUCTURAL_DIM, torch.long),
            rows_or_empty(pokemon_dynamic, POKEMON_DYNAMIC_DIM, torch.float32),
            rows_or_empty(attack_dynamic, ATTACK_DYNAMIC_DIM, torch.float32),
            torch.tensor(offsets, dtype=torch.long))


def sparse_tensors(vector):
    return (
        torch.tensor(vector.index, dtype=torch.int64),
        torch.tensor(vector.value, dtype=torch.float32),
        torch.tensor(vector.offset, dtype=torch.int64),
    )


def dense_tensor(values):
    return torch.tensor(values, dtype=torch.float32).unsqueeze(0)


def load_checkpoint(path):
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location='cpu')
    if not isinstance(checkpoint, dict) or 'model' not in checkpoint or 'config' not in checkpoint:
        raise ValueError('Expected checkpoint with model and config keys')
    return checkpoint


def read_model_manifest():
    with open(asset_path('model_manifest.json'), encoding='utf-8') as handle:
        manifest = json.load(handle)
    enabled = manifest.get('ensemble_enabled')
    damage_counter_ko_mask = manifest.get('damage_counter_ko_mask')
    model_files = manifest.get('model_files')
    if type(enabled) is not bool:
        raise ValueError('model_manifest ensemble_enabled must be boolean')
    if type(damage_counter_ko_mask) is not bool:
        raise ValueError('model_manifest damage_counter_ko_mask must be boolean')
    if not isinstance(model_files, list) or not all(
        isinstance(name, str) and name for name in model_files
    ):
        raise ValueError('model_manifest model_files must contain names')
    if len(set(model_files)) != len(model_files):
        raise ValueError('model_manifest model_files must be distinct')
    if enabled and len(model_files) < 2:
        raise ValueError('enabled ensemble requires at least two models')
    if not enabled and len(model_files) != 1:
        raise ValueError('disabled ensemble requires exactly one model')
    return enabled, damage_counter_ko_mask, model_files


def load_models():
    ensemble_enabled, damage_counter_ko_mask, model_files = read_model_manifest()
    cards = all_card_data()
    attacks = all_attack()
    models = []
    reference_config = catalog = card_feature_table = attack_feature_table = None
    compatibility_keys = ('card_count', 'attack_count', 'encoder_size')
    for model_index, model_file in enumerate(model_files, start=1):
        checkpoint = load_checkpoint(asset_path(model_file))
        config = ModelConfig(**checkpoint['config'])
        if reference_config is None:
            reference_config = config
            catalog = build_numeric_catalog(cards, attacks, config.card_count)
            card_feature_table = catalog[0]
            attack_feature_table = build_attack_feature_table(
                attacks, config.attack_count
            )
        else:
            differences = [
                key for key in compatibility_keys
                if getattr(config, key) != getattr(reference_config, key)
            ]
            if differences:
                raise ValueError(
                    f'Model {model_index} is incompatible: {differences}'
                )
        model = PTCGTransformer(
            config, card_feature_table, attack_feature_table
        )
        model.load_state_dict(checkpoint['model'])
        model.eval()
        models.append(model)
        del checkpoint
    return tuple(models), reference_config, catalog, ensemble_enabled, damage_counter_ko_mask


MODELS, CONFIG, NUMERIC_CATALOG, ENSEMBLE_ENABLED, DAMAGE_COUNTER_KO_MASK = load_models()
ACTION_HISTORY = deque(maxlen=HISTORY_STEPS)
REVEALED_HANDS = RevealedHandTracker()


def agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    if obs.select is None:
        ACTION_HISTORY.clear()
        REVEALED_HANDS.reset()
        return MY_DECK
    REVEALED_HANDS.update(obs.logs)
    own_revealed, opponent_revealed = REVEALED_HANDS.relative_cards(
        obs.current.yourIndex
    )

    actions = enumerate_actions(
        len(obs.select.option), obs.select.minCount, obs.select.maxCount
    )
    if not actions:
        selected = list(range(min(obs.select.maxCount, len(obs.select.option))))
        fallback = decoder_features(obs, [selected], CONFIG.card_count, CONFIG.attack_count, NUMERIC_CATALOG)
        ACTION_HISTORY.append(completed_history_action(obs, selected, fallback))
        return selected
    action_eligible = damage_counter_action_eligibility(obs, actions)
    encoder, pokemon_appear, own, opponent, global_state, revealed = encoder_features(
        obs, MY_DECK, CONFIG.card_count, NUMERIC_CATALOG,
        own_revealed, opponent_revealed,
    )
    option_categorical, option_numeric, pokemon_dynamic, attack_dynamic, action_index, action_offset = decoder_features(
        obs, actions, CONFIG.card_count, CONFIG.attack_count, NUMERIC_CATALOG
    )
    history_inputs = history_tensors(ACTION_HISTORY)
    model_inputs = (
        *sparse_tensors(encoder),
        torch.tensor(pokemon_appear, dtype=torch.long).unsqueeze(0),
        dense_tensor(own), dense_tensor(opponent),
        dense_tensor(global_state),
        torch.tensor(revealed, dtype=torch.long).unsqueeze(0),
        *history_inputs, option_categorical, option_numeric,
        pokemon_dynamic, attack_dynamic, action_index, action_offset,
    )
    with torch.inference_mode():
        if ENSEMBLE_ENABLED:
            probability_sum = None
            expected_shape = None
            for model in MODELS:
                policy_logits = model(*model_inputs)
                if expected_shape is None:
                    expected_shape = policy_logits.shape
                elif policy_logits.shape != expected_shape:
                    raise ValueError('Ensemble models returned different action shapes')
                probabilities = torch.softmax(policy_logits.float(), dim=1)
                probability_sum = (
                    probabilities if probability_sum is None
                    else probability_sum + probabilities
                )
            policy_scores = probability_sum / len(MODELS)
        else:
            policy_scores = MODELS[0](*model_inputs)
    if DAMAGE_COUNTER_KO_MASK:
        eligible = torch.from_numpy(action_eligible).to(policy_scores.device)
        policy_scores = policy_scores.masked_fill(~eligible.unsqueeze(0), float('-inf'))
    selected = actions[int(policy_scores[0].argmax().item())]
    decoded = (option_categorical, option_numeric, pokemon_dynamic, attack_dynamic, action_index, action_offset)
    ACTION_HISTORY.append(completed_history_action(obs, selected, decoded))
    return selected


In [ ]:
# Syntax-check the generated agent without importing it yet.
compile(Path('main.py').read_text(), 'main.py', 'exec')
print('main.py syntax OK')

In [ ]:
import tarfile

with tarfile.open('submission.tar.gz', 'w:gz') as tar:
    tar.add('main.py', arcname='main.py')
    tar.add('deck.csv', arcname='deck.csv')
    tar.add('model_manifest.json', arcname='model_manifest.json')
    for model_path, archive_name in zip(MODEL_PATHS, MODEL_ARCHIVE_NAMES):
        tar.add(model_path, arcname=archive_name)
    tar.add(CG_PATH, arcname='cg')

size_mb = Path('submission.tar.gz').stat().st_size / 1024**2
print(f'Created submission.tar.gz ({size_mb:.2f} MB)')
if size_mb > 197.7:
    print(f'WARNING: submission exceeds Kaggle limit by {size_mb - 197.7:.2f} MiB')
with tarfile.open('submission.tar.gz', 'r:gz') as tar:
    print('Archive root:', sorted({name.split('/')[0] for name in tar.getnames()}))